In [1]:
!pip install tensorflow gradio joblib rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 109.7 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [3]:
# Load dataset
df = pd.read_csv("/content/dataset_1200variants_per_disease.csv")

target_col = "Disease"
symptom_cols = [c for c in df.columns if c != target_col]

# Clean NaNs
df[symptom_cols] = df[symptom_cols].where(df[symptom_cols].notna(), None)

print("Dataset shape:", df.shape)

/tmp/ipykernel_537/2485496246.py:2: DtypeWarning: Columns (8,9,10,11,12,13,14,15,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/dataset_1200variants_per_disease.csv")


Dataset shape: (54120, 18)


In [4]:
all_symptoms = set()
for col in symptom_cols:
    all_symptoms.update(df[col].dropna().unique())

all_symptoms = sorted(all_symptoms)
symptom_index = {s: i for i, s in enumerate(all_symptoms)}

print("Total symptoms:", len(symptom_index))

Total symptoms: 131


In [5]:
def encode_symptoms(symptoms):
    vector = np.zeros(len(symptom_index))
    for s in symptoms:
        if s in symptom_index:
            vector[symptom_index[s]] = 1
    return vector

In [6]:
# Build X and y
X = []
for _, row in df.iterrows():
    symptoms = [row[c] for c in symptom_cols if row[c] is not None]
    X.append(encode_symptoms(symptoms))

X = np.array(X)

disease_encoder = LabelEncoder()
y = disease_encoder.fit_transform(df[target_col])

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (54120, 131)
y shape: (54120,)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    Dense(256, activation="relu", input_shape=(X.shape[1],)),
    Dropout(0.3),
    Dense(128, activation="relu"),
    Dropout(0.2),
    Dense(len(disease_encoder.classes_), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │        33,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 41)             │         5,289 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 71,977 (281.16 KB)

 Trainable params: 71,977 (281.16 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/50
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.6840 - loss: 1.4462 - val_accuracy: 0.8734 - val_loss: 0.3173
Epoch 2/50
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8731 - loss: 0.3452 - val_accuracy: 0.8776 - val_loss: 0.3041
Epoch 3/50
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8785 - loss: 0.3175 - val_accuracy: 0.8796 - val_loss: 0.2929
Epoch 4/50
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8845 - loss: 0.3046 - val_accuracy: 0.8815 - val_loss: 0.2897
Epoch 5/50
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8809 - loss: 0.3032 - val_accuracy: 0.8819 - val_loss: 0.2845
Epoch 6/50
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8814 - loss: 0.2905 - val_accuracy: 0.8829 - val_loss: 0.2840
Epoch 7/50
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8822 - loss: 0.2896 - val_accuracy: 0.8815 - val_loss: 0.2899
Epoch 8/50
1083/1083 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8878 - loss: 0.2807 - 

In [10]:
loss, acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {acc:.4f}")

339/339 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8908 - loss: 0.2450
Test Accuracy: 0.8917


In [11]:
model.save("disease_model.keras")
joblib.dump(symptom_index, "symptom_index.pkl")
joblib.dump(disease_encoder, "disease_encoder.pkl")

print("✅ Model & encoders saved")

✅ Model & encoders saved


In [12]:
import gradio as gr
from tensorflow.keras.models import load_model
from rapidfuzz import process, fuzz

In [13]:
model = load_model("disease_model.keras")
symptom_index = joblib.load("symptom_index.pkl")
disease_encoder = joblib.load("disease_encoder.pkl")

symptom_list = list(symptom_index.keys())

In [14]:
def normalize_symptom(s):
    return s.lower().strip().replace("_", " ")

def match_symptom(user_symptom, threshold=80):
    match, score, _ = process.extractOne(
        user_symptom,
        symptom_list,
        scorer=fuzz.ratio
    )
    return match if score >= threshold else None

In [15]:
symptom_list = sorted(symptom_index.keys())

In [16]:
def predict_disease(symptom_text):
    raw_inputs = symptom_text.split(",")

    vector = np.zeros(len(symptom_index))
    recognized, ignored = [], []

    for raw in raw_inputs:
        norm = normalize_symptom(raw)
        matched = match_symptom(norm)

        if matched:
            vector[symptom_index[matched]] = 1
            recognized.append(matched)
        else:
            ignored.append(raw.strip())

    if vector.sum() == 0:
        return "⚠️ No valid symptoms recognized."

    preds = model.predict(vector.reshape(1, -1))[0]
    idx = np.argmax(preds)

    disease = disease_encoder.inverse_transform([idx])[0]
    confidence = preds[idx] * 100

    return (
        f"🩺 Disease: {disease}\n"
        f"📊 Confidence: {confidence:.2f}%\n\n"
        f"✅ Recognized: {', '.join(recognized)}\n"
        f"❌ Ignored: {', '.join(ignored) if ignored else 'None'}"
    )

In [17]:
def predict_disease_dropdown(selected_symptoms, other_symptoms):
    if not selected_symptoms:
        return "⚠️ Please select at least 2 symptoms."

    if len(selected_symptoms) < 2:
        return "⚠️ At least 2 symptoms are required for reliable prediction."

    vector = np.zeros(len(symptom_index))
    for s in selected_symptoms:
        vector[symptom_index[s]] = 1

    probs = model.predict(vector.reshape(1, -1))[0]
    top_indices = probs.argsort()[-3:][::-1]

    output = "🩺 Top Possible Diseases:\n\n"
    for i, idx in enumerate(top_indices, start=1):
        disease = disease_encoder.inverse_transform([idx])[0]
        confidence = probs[idx] * 100
        output += f"{i}. {disease} — {confidence:.2f}%\n"

    output += "\n✅ Selected Symptoms:\n"
    output += ", ".join(selected_symptoms)

    if other_symptoms.strip():
        output += (
            "\n\n📝 Other Symptoms (not used in prediction):\n"
            + other_symptoms
        )

    return output

In [18]:
gr.Interface(
    fn=predict_disease_dropdown,
    inputs=[
        gr.Dropdown(
            choices=symptom_list,
            multiselect=True,
            label="Select symptoms",
            info="Search and select all applicable symptoms"
        ),
        gr.Textbox(
            label="Other symptoms (optional)",
            placeholder="e.g., feeling cold, anxiety",
            lines=2
        )
    ],
    outputs=gr.Textbox(
        label="Prediction",
        lines=12
    ),
    title="Disease Diagnosis System",
    description="Select symptoms from the list for accurate prediction"
).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8301941f0c76955ee9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
